<a href="https://colab.research.google.com/github/harshita10sharma/Youtube-video-dubbing-system/blob/main/Youtube_Dubbing_IndicTrans2_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q transformers sentencepiece torch IndicTransToolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.1/546.1 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import transformers
import sentencepiece
from IndicTransToolkit.processor import IndicProcessor

print("Transformers:", transformers.__version__)
print("IndicTransToolkit: OK")
print("SentencePiece: OK")

Transformers: 4.53.2
IndicTransToolkit: OK
SentencePiece: OK


In [6]:
from huggingface_hub import login

login()

In [7]:
from huggingface_hub import whoami

info = whoami()
print("Logged in as:", info["name"])

Logged in as: harshita10sh


In [8]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "ai4bharat/indictrans2-indic-en-dist-200M"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading IndicTrans2...")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
).to(device)

print("IndicTrans2 loaded successfully!")

Loading IndicTrans2...
Device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

IndicTrans2 loaded successfully!


In [9]:
from IndicTransToolkit.processor import IndicProcessor

ip = IndicProcessor(inference=True)

source_text = [
    "नमस्ते, मेरा नाम हर्षिता है।",
    "मैं मशीन लर्निंग और आर्टिफिशियल इंटेलिजेंस पर काम कर रही हूँ।",
    "यह एक वीडियो डबिंग सिस्टम है।"
]

# IndicTrans2 language codes
src_lang = "hin_Deva"
tgt_lang = "eng_Latn"

# Prepare input
batch = ip.preprocess_batch(
    source_text,
    src_lang=src_lang,
    tgt_lang=tgt_lang
)

# Tokenize
inputs = tokenizer(
    batch,
    padding=True,
    truncation=True,
    return_tensors="pt"
).to(device)

# Generate translation
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        num_beams=5,
        num_return_sequences=1,
        max_length=256
    )

# Decode
generated_text = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)

# Post-process
translations = ip.postprocess_batch(
    generated_text,
    lang=tgt_lang
)

print("\nHindi → English\n")
for src, tgt in zip(source_text, translations):
    print("Hindi :", src)
    print("English:", tgt)
    print()


Hindi → English

Hindi : नमस्ते, मेरा नाम हर्षिता है।
English: Hi, my name is Harshita.

Hindi : मैं मशीन लर्निंग और आर्टिफिशियल इंटेलिजेंस पर काम कर रही हूँ।
English: I am working on machine learning and artificial intelligence.

Hindi : यह एक वीडियो डबिंग सिस्टम है।
English: It is a video dubbing system.



In [12]:
import json

with open("transcript.json", "r", encoding="utf-8") as f:
    transcript = json.load(f)

print("Number of Whisper segments:", len(transcript))

print("\nFirst 5 segments:\n")

for i, segment in enumerate(transcript[:5]):
    print(f"[{i}]")
    print("Start :", segment["start"])
    print("End   :", segment["end"])
    print("Text  :", segment["text"])
    print()

Number of Whisper segments: 260

First 5 segments:

[0]
Start : 10.06
End   : 14.06
Text  : कहानिया हम सब की जिन्दिगी में बहुत महतुपून होती है, बहुत इंपूरेंट होती हैं।

[1]
Start : 14.06
End   : 16.06
Text  : यह बात आप समझते हैं, जानते हैं।

[2]
Start : 16.06
End   : 19.06
Text  : आप सोचीए वो सारी कहानिया जो हमने बचपं में स्कूलो में पढडली,

[3]
Start : 19.06
End   : 21.06
Text  : हमारे माबाप नहीं हमें सुना दी।

[4]
Start : 21.06
End   : 23.06
Text  : हमारे दोस्तों से हमने कुछ सीखली

